In [43]:
import pandas as pd
import numpy as np

In [44]:
a = np.array([1,2,3,4])
print(a)

[1 2 3 4]


In [45]:
df = pd.Series(a)
print(df)

0    1
1    2
2    3
3    4
dtype: int64


In [46]:
b = np.array([[1,2,3,4],[5,6,7,8]])
df2 = pd.DataFrame(b)
print(b)
print(df2)

[[1 2 3 4]
 [5 6 7 8]]
   0  1  2  3
0  1  2  3  4
1  5  6  7  8


In [47]:
c = np.array([[[1,2,3,4],[5,6,7,8]],[[9,10,11,12],[13,14,15,16]]])
print(c)


[[[ 1  2  3  4]
  [ 5  6  7  8]]

 [[ 9 10 11 12]
  [13 14 15 16]]]


In [48]:
df = pd.read_csv(r"C:\Users\Nidho\Downloads\Traning Dataset-001\Traning Dataset\Bengaluru_House_Data.csv")
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [49]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  str    
 1   availability  13320 non-null  str    
 2   location      13319 non-null  str    
 3   size          13304 non-null  str    
 4   society       7818 non-null   str    
 5   total_sqft    13320 non-null  str    
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), str(6)
memory usage: 936.7 KB


In [50]:
df.isnull().sum()

area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64

In [51]:
location_avg_price = (
    df.groupby("location")["price"]
    .mean()
    .rename("location_avg_price")
)

df = df.join(location_avg_price, on="location")

print(df[["location", "price", "location_avg_price"]].head())

                   location   price  location_avg_price
0  Electronic City Phase II   39.07           48.316667
1          Chikka Tirupathi  120.00          115.015882
2               Uttarahalli   62.00           61.252527
3        Lingadheeranahalli   95.00          115.348261
4                  Kothanur   51.00           95.630303


In [52]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   area_type           13320 non-null  str    
 1   availability        13320 non-null  str    
 2   location            13319 non-null  str    
 3   size                13304 non-null  str    
 4   society             7818 non-null   str    
 5   total_sqft          13320 non-null  str    
 6   bath                13247 non-null  float64
 7   balcony             12711 non-null  float64
 8   price               13320 non-null  float64
 9   location_avg_price  13319 non-null  float64
dtypes: float64(4), str(6)
memory usage: 1.0 MB


In [53]:
print(location_avg_price.describe())

print(
    df.groupby("location")["price"]
    .agg(["count", "mean", "median", "min", "max"])
    .sort_values("mean")
    .head(10)
)

count    1305.000000
mean      135.218127
std       144.911422
min        15.000000
25%        61.000000
50%        91.875000
75%       150.000000
max      1900.000000
Name: location_avg_price, dtype: float64
                                 count    mean  median   min    max
location                                                           
Alur                                 3  15.000  15.000  10.0  20.00
Makali                               3  16.000  15.000  15.0  18.00
 Anekal                              1  16.000  16.000  16.0  16.00
BAGUR                                1  17.000  17.000  17.0  17.00
Celebrity Paradise Layout            2  19.245  19.245  16.0  22.49
Bilal Nagar                          1  20.000  20.000  20.0  20.00
Kengeri Satellite Town Stage II      1  20.000  20.000  20.0  20.00
Venkateswara Nagar                   1  20.000  20.000  20.0  20.00
Pillanna Gardens                     1  20.000  20.000  20.0  20.00
Shauhardha Layout                    1  20.

In [54]:
# Calculate quartile boundaries from location-level average prices
price_levels = location_avg_price.dropna()

q1 = price_levels.quantile(0.25)
q2 = price_levels.quantile(0.50)
q3 = price_levels.quantile(0.75)

def classify_price(price):
    if pd.isna(price):
        return np.nan
    elif price <= q1:
        return "Budget"
    elif price <= q2:
        return "Mid-Range"
    elif price <= q3:
        return "Premium"
    else:
        return "Luxury"

df["estimated_price_range"] = df["location_avg_price"].apply(classify_price)

print(
    df[["location", "price", "society",
        "location_avg_price", "estimated_price_range"]].head(15)
)

                    location   price  society  location_avg_price  \
0   Electronic City Phase II   39.07  Coomee            48.316667   
1           Chikka Tirupathi  120.00  Theanmp          115.015882   
2                Uttarahalli   62.00      NaN           61.252527   
3         Lingadheeranahalli   95.00  Soiewre          115.348261   
4                   Kothanur   51.00      NaN           95.630303   
5                 Whitefield   38.00  DuenaTa          128.012324   
6           Old Airport Road  204.00  Jaades           184.080303   
7               Rajaji Nagar  600.00  Brway G          327.693084   
8               Marathahalli   63.25      NaN          102.213714   
9               Gandhi Bazar  370.00      NaN          240.000000   
10                Whitefield   70.00      NaN          128.012324   
11                Whitefield  295.00  Prrry M          128.012324   
12        7th Phase JP Nagar   38.00  Shncyes          100.522953   
13                 Gottigere   40.

In [55]:
price_levels = location_avg_price.dropna()

q1 = price_levels.quantile(0.25)
q2 = price_levels.quantile(0.50)
q3 = price_levels.quantile(0.75)

def classify_price(price):
    if pd.isna(price):
        return np.nan
    elif price <= q1:
        return "Budget"
    elif price <= q2:
        return "Mid-Range"
    elif price <= q3:
        return "Premium"
    else:
        return "Luxury"

df["estimated_price_range"] = df["location_avg_price"].apply(classify_price)

print(
    df[["location", "price", "society",
        "location_avg_price", "estimated_price_range"]].head(15)
)

                    location   price  society  location_avg_price  \
0   Electronic City Phase II   39.07  Coomee            48.316667   
1           Chikka Tirupathi  120.00  Theanmp          115.015882   
2                Uttarahalli   62.00      NaN           61.252527   
3         Lingadheeranahalli   95.00  Soiewre          115.348261   
4                   Kothanur   51.00      NaN           95.630303   
5                 Whitefield   38.00  DuenaTa          128.012324   
6           Old Airport Road  204.00  Jaades           184.080303   
7               Rajaji Nagar  600.00  Brway G          327.693084   
8               Marathahalli   63.25      NaN          102.213714   
9               Gandhi Bazar  370.00      NaN          240.000000   
10                Whitefield   70.00      NaN          128.012324   
11                Whitefield  295.00  Prrry M          128.012324   
12        7th Phase JP Nagar   38.00  Shncyes          100.522953   
13                 Gottigere   40.

In [56]:
missing_society = df["society"].isna()

df.loc[missing_society, "society_estimate"] = (
    df.loc[missing_society, "estimated_price_range"]
)

# Preserve known society names
df.loc[~missing_society, "society_estimate"] = (
    df.loc[~missing_society, "society"]
)

print(df.loc[
    missing_society,
    ["location", "price", "location_avg_price",
     "estimated_price_range", "society_estimate"]
].head(20))

                    location   price  location_avg_price  \
2                Uttarahalli   62.00           61.252527   
4                   Kothanur   51.00           95.630303   
8               Marathahalli   63.25          102.213714   
9               Gandhi Bazar  370.00          240.000000   
10                Whitefield   70.00          128.012324   
13                 Gottigere   40.00           61.017292   
19        Manayata Tech Park   48.00           81.875000   
20                   Kengeri   15.00           61.441507   
23                 Bellandur  103.00           91.193333   
25           Mangammanapalya   56.00           51.833333   
27                Whitefield   81.00          128.012324   
36  Electronics City Phase 1   45.50           55.101818   
38            Chikkalasandra   56.12           71.170333   
39               Uttarahalli   45.00           61.252527   
43                 Yelahanka   48.00           86.098662   
45                HSR Layout  200.00    

In [57]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   area_type              13320 non-null  str    
 1   availability           13320 non-null  str    
 2   location               13319 non-null  str    
 3   size                   13304 non-null  str    
 4   society                7818 non-null   str    
 5   total_sqft             13320 non-null  str    
 6   bath                   13247 non-null  float64
 7   balcony                12711 non-null  float64
 8   price                  13320 non-null  float64
 9   location_avg_price     13319 non-null  float64
 10  estimated_price_range  13319 non-null  str    
 11  society_estimate       13320 non-null  str    
dtypes: float64(4), str(8)
memory usage: 1.2 MB


In [58]:
df.to_csv(r"C:\Users\Nidho\Downloads\Traning Dataset-001\Traning Dataset\Practice1.csv")

In [65]:
df.isna().sum()

area_type                   0
availability                0
location                    0
size                       16
society                  5502
total_sqft                  0
bath                       73
balcony                   609
price                       0
location_avg_price          0
estimated_price_range       0
society_estimate            0
dtype: int64

In [63]:
df = df.dropna(subset=["location"])

In [66]:
df["balcony"] = df["balcony"].fillna(df["balcony"].median())

In [67]:
df.isna().sum()

area_type                   0
availability                0
location                    0
size                       16
society                  5502
total_sqft                  0
bath                       73
balcony                     0
price                       0
location_avg_price          0
estimated_price_range       0
society_estimate            0
dtype: int64

In [69]:
# Calculate the mode of bath for each society
society_bath_mode = (
    df.groupby("society")["bath"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
)

print(society_bath_mode.head(10))

society
3Codeli    2.0
7 ise P    2.0
A idse     2.0
A rtsai    2.0
ACersd     2.0
ACiteda    2.0
ACopsva    2.0
AFe 3Ph    4.0
AGoutLa    2.0
AHavenc    5.0
Name: bath, dtype: float64


In [70]:
# Map each society's mode to its rows
bath_mode_by_society = df["society"].map(society_bath_mode)

# Fill missing bath values using the society-specific mode
df["bath"] = df["bath"].fillna(bath_mode_by_society)

In [72]:
df = df.dropna(subset=["bath"])

In [74]:
df = df.dropna(subset=['size'])

In [75]:
df.isna().sum()

area_type                   0
availability                0
location                    0
size                        0
society                  5499
total_sqft                  0
bath                        0
balcony                     0
price                       0
location_avg_price          0
estimated_price_range       0
society_estimate            0
dtype: int64